In [ ]:
# --- repo path bootstrap ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, derived, require)
from utils.panels import save_panel

import pandas as pd
import numpy as np
import os

# Pycytominer
from pycytominer import feature_select
from pycytominer import normalize
# from pycytominer import aggregate

# Set current working directory


In [ ]:
def list_features(df):
    # List features
    list_of_selected_features = list(df.columns.values)
    list_of_metadata = list(df.columns[df.columns.str.contains("Metadata_")])
    list_of_selected_features = list(set(list_of_selected_features) - set(list_of_metadata))
    
    return list_of_selected_features, list_of_metadata

In [ ]:
cell_line = 'HT29' # HT29 or HCT116
data_type = 'aggregates'

In [ ]:
# List all files in directory
dir = str(features("exp1_main", "SingleSlice")) + "/"

files = os.listdir(dir)

name = dir

# Select all files with HCT116 in the name as well as MedianAgg_meanstd
files = [file for file in files if cell_line in file and 'MedianAgg' in file]


In [ ]:
# Load the parquet file into a pandas dataframe

# Load all files
data = []
for file in files:
    data.append(pd.read_parquet(dir + file))

data = pd.concat(data)

In [ ]:
## Prepare the metadata
dataset = data.copy()

# Normalize each slice for each plate separately
dataset["Metadata_plate_slice"] = (
    dataset["Metadata_Barcode"] + "_" + dataset["Metadata_Site"].astype(str)
    )

In [ ]:
#
# Normalize separately per 1) plate and 2) cell line
#

units = dataset["Metadata_plate_slice"].unique() # Per slice in each plate

# Itnitialize an empty dataframe
normalized = pd.DataFrame(columns=dataset.columns.values)

for unit in units:
    
    print(unit)
    annotated_temp = dataset[dataset['Metadata_plate_slice'] == unit]

    # Normalize: choose between standardize, robustize, mad_robustize, spherize 
    normalized_temp = normalize(annotated_temp, 
                                features=list_features(dataset)[0],image_features=False, 
                                meta_features="infer", samples="Metadata_cmpdname == 'dmso'", 
                                method="standardize")
    normalized = pd.concat([normalized, normalized_temp], ignore_index=True)

    


In [ ]:
# Aggregate profiles across z-slices

features = list_features(normalized)[0]
metadata_cols = [col for col in normalized.columns if col not in features + ['Metadata_Site', 'Metadata_PlateWell','Metadata_plate_slice']]

aggregated_df = normalized.groupby(['Metadata_PlateWell']).agg(
    {**{col: 'first' for col in metadata_cols},  # Keep the first occurrence of metadata columns
    **{col: 'median' for col in features}}  # Aggregate features by mean (or any other function)
).reset_index()

In [ ]:
# Feature selection: "variance_threshold", "correlation_threshold", "drop_na_columns", "blocklist", "drop_outliers", "noise_removal",
# to_clip_df = feature_select(aggregated_df, features=list_features(normalized)[0], operation=["variance_threshold", "correlation_threshold","drop_na_columns", "blocklist"])
to_clip_df = feature_select(aggregated_df, features=list_features(normalized)[0], operation=["variance_threshold", "correlation_threshold","drop_na_columns"])
# Instead of removing the outliers, we can clip them
selected_df = pd.concat([to_clip_df[list_features(to_clip_df)[1]], to_clip_df[list_features(to_clip_df)[0]].clip(lower=-40, upper=40, axis=1)], axis=1)
print(selected_df.shape)

In [ ]:
# Save the data
OutputDir = str(derived("exp1_main")) + "/"   # derived/, never the deposit
if not os.path.exists(OutputDir): 
    os.makedirs(OutputDir)

# Save as parquet
selected_df.to_parquet(('{}selected_data_{}_{}.parquet').format(OutputDir, data_type, cell_line))
